# CRS correlator walkthrough: corr8 / corr32 / corr64

One notebook for the three firmware modes. Pick `MODE` in the first code cell; everything
else is identical. The flow is:

1. program the boards and initialize the array (`FPGAArray`)
2. capture raw **ADC** time streams from a few inputs and plot them
3. capture raw **FFT** frames (full 32+32-bit resolution, pre-scaler) and plot spectra
4. compute and program the **digital gains** that quantize the FFT to the 4+4-bit correlator input
5. start the **correlator** and read **visibilities**
6. overplot ADC-derived spectrum, firmware FFT, quantized FFT prediction and correlator autocorrelation

This follows `crs_corr64_tests.ipynb` (the reference test) with the ADC capture and gain
computation taken from `corriscope.pocket_correlator`. Lines marked `# verify on hardware`
are scalings that were written without a board in front of the author.

Requirements: jumbo frames and large UDP buffers on the capture interface (see
`python -m corriscope.network_utils --status`), the bitstreams in
`corriscope/fpga_firmware/bitstreams/` (git-lfs), `%matplotlib ipympl` needs `ipympl`.

In [ ]:
import logging, time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib ipympl
plt.rcParams['figure.figsize'] = (11, 5)
logging.getLogger().setLevel(logging.INFO)

from corriscope.fpga_array import FPGAArray

# ---- pick the firmware mode and the hardware map ------------------------------------
MODE = 'corr8'     # 'corr8' | 'corr32' | 'corr64'

# Hardware maps. 'crs *:N' is the board in crate slot N, whatever its serial number.
# With more than one crate on the network, name the boards instead: 'crs 127:1 134:2 ...'.
SYNC_TRIG = dict(sync_method='trig', sync_source='bp_trig',
                 sync_master=1, sync_master_source='sync_out', sync_master_output=1)
SETUP = {
    # one board in slot 1; software sync
    'corr8':  dict(hwm='crs *:1', sync_method='local'),
    # four boards in slots 1-4 (one backplane); the board in slot 1 sends the trigger
    'corr32': dict(hwm='crs *:1 *:2 *:3 *:4', **SYNC_TRIG),
    # eight boards in slots 1-8 (two backplanes with SFP links); the board in slot 1 sends the trigger
    'corr64': dict(hwm='crs *:1 *:2 *:3 *:4 *:5 *:6 *:7 *:8', **SYNC_TRIG),
}

## 1. Program and initialize

`mode=` selects the bitstream (`chfpga_crs_<mode>.bit`) and the array initialization.
`prog` defaults to 2 (always reprogram) when `mode` is given; pass `prog=1` to keep an
already-programmed FPGA. `bitfile='/some/folder'` overrides the bitstream search folder.

In [ ]:
ca = await FPGAArray(mode=MODE, stderr_log_level='info', **SETUP[MODE])

fs = ca.ib[0]._sampling_frequency          # ADC sampling frequency (3.2 GHz on CRS)
frame_len = ca.ib[0].FRAME_LENGTH          # ADC samples per frame
nbins = ca.ib[0].NUMBER_OF_FREQUENCY_BINS  # FFT bins per frame (frame_len / 2)
nchan = ca.ib[0].NUMBER_OF_CHANNELIZERS    # inputs per board (8)
print(f'{len(ca.ib)} board(s): {[str(b) for b in ca.ib]}')
print(f'firmware {await ca.ib[0].get_fpga_firmware_timestamp_async()}, fs={fs/1e6:.0f} MHz, '
      f'{frame_len} samples/frame, {nbins} bins, frame period {ca.ib[0].FRAME_PERIOD*1e6:.2f} us')
print('SYNC counters (should all be equal):', list(ca.ib.REFCLK.SYNC_CTR))

Input labels: the receivers return data for every board in `ca.ib` order and every
channel on that board, so a flat input index is `board_index * 8 + channel`.

In [ ]:
def input_labels(channels=range(nchan)):
    return [f'{b.get_string_id()} ch{ch}' for b in ca.ib for ch in channels]

labels = input_labels()
print(len(labels), 'inputs:', labels[:4], '...')

## 2. Raw ADC time streams

UCAP (the capture engine) taps the FUNCGEN output, which carries the ADC samples when the
data source is `'adc'`. UCAP mode 0 sends 2 consecutive frames of all 8 channels per burst;
mode 3 sends 16 consecutive frames of a single channel. The raw receiver returns 14-bit
signed integers (the packet processor removes the 16-bit bus padding).

In [ ]:
await ca.set_channelizers_async(data_source='adc', funcgen_left_shift=0, funcgen_right_shift=0,
                                fft_bypass=False, scaler_bypass=False,
                                offset_binary_encoding=False, sync=False)
await ca.start_data_capture_async(0.02, source='funcgen', mode=0, data_type=None, sync=False)

r = await ca.get_data_receiver_async(tcp=False)
t_adc, adc, cnt = r.read_raw_frames(ncap=4, split=True, flush=True)   # adc: (n_inputs, n_frames, frame_len)
print(f'ADC data {adc.shape} dtype={adc.dtype}  min={adc.min()} max={adc.max()}  '
      f'(14-bit range is -8192..8191)')

In [ ]:
inputs_to_plot = [0, 1, 2, 3]       # flat input indices (board*8 + channel)
nsamp = 2000
tt = np.arange(nsamp) / fs * 1e9     # ns

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 4.5))
for i in inputs_to_plot:
    ax0.plot(tt, adc[i, 0, :nsamp], lw=0.8, label=labels[i])
    ax1.hist(adc[i].ravel(), bins=128, histtype='step', label=f'{labels[i]}  rms={adc[i].std():.1f}')
ax0.set(xlabel='time [ns]', ylabel='ADC code (14 bit)', title='raw ADC samples, first frame'); ax0.grid(); ax0.legend(fontsize=8)
ax1.set(xlabel='ADC code', ylabel='count', title='sample histogram'); ax1.set_yscale('log'); ax1.grid(); ax1.legend(fontsize=8)

In [ ]:
# rms per input across the whole array (quick health check of every ADC)
rms = adc.reshape(adc.shape[0], -1).std(axis=1)
plt.figure(figsize=(12, 3.5))
plt.bar(range(len(rms)), rms)
plt.xticks(range(len(rms)), labels, rotation=90, fontsize=7)
plt.ylabel('rms [ADC codes]'); plt.title('ADC rms per input'); plt.grid(axis='y')

## 3. Raw FFT frames

`data_type='fft_2x'` captures the FFT output before the gain, with 32+32 bits. One frame
carries the even bins and the next frame the odd bins. The packet processor puts them back
in frequency order. UCAP `mode=3` sends 16 consecutive frames of one input per burst, on
every board at the same time. The cell captures the 8 inputs one after the other. This is
the capture that the reference test used; UCAP `mode=0` returns 16384 values per frame with
this data type and is not used here.

In [ ]:
freq = np.arange(nbins) * fs / 2 / nbins / 1e6      # MHz, first Nyquist zone; bin spacing is fs / 2 / nbins

# fft_2x with UCAP mode 3: 16 consecutive frames of one input per burst, on every board at once.
# Capture each of the 8 inputs in turn and stack them in flat input order (board * 8 + channel).
fft_rows = {}
for ch in range(nchan):
    await ca.start_data_capture_async(0.05, source='scaler', mode=3, channels=[ch], data_type='fft_2x', sync=False)
    _, d, _ = r.read_raw_frames(ncap=2, split=True, flush=True)   # d: (n_boards, 32 frames, nbins) complex
    for b in range(len(ca.ib)):
        fft_rows[b * nchan + ch] = d[b]
fft = np.stack([fft_rows[i] for i in range(len(labels))])          # (inputs, frames, nbins) complex
print(f'FFT data {fft.shape} {fft.dtype}')
assert fft.shape[-1] == nbins, f'expected {nbins} bins per frame, got {fft.shape[-1]}'
fft_pow = (np.abs(fft) ** 2).mean(axis=1)                          # mean |X|^2 per input and bin

plt.figure(figsize=(12, 5))
for i in inputs_to_plot:
    plt.semilogy(freq[1:], fft_pow[i, 1:], lw=0.7, label=labels[i])
plt.xlabel('frequency [MHz]'); plt.ylabel('|FFT|$^2$ (32+32-bit units)'); plt.title('firmware FFT, mean power spectrum')
plt.grid(); plt.legend(fontsize=8)

### ADC-derived spectrum (model of the firmware PFB/FFT)

`FFT.pfb_fft` is the Python model of the firmware 4-tap polyphase filter bank + FFT. It
needs 4 consecutive frames, so capture one channel with UCAP mode 3 (16 consecutive frames)
and compare with the firmware FFT of the same channel. The firmware divides by 2 for every
set bit of the FFT shift schedule.

In [ ]:
model_input = inputs_to_plot[0]
mb, ch = ca.ib[model_input // nchan], model_input % nchan

await ca.start_data_capture_async(0.02, source='funcgen', mode=3, channels=[ch], data_type=None, sync=False)
_, adc16, _ = r.read_raw_frames(ncap=2, split=True, flush=True)       # (n_boards, 32, frame_len)
frames = adc16[model_input // nchan].astype(float)

fft_shift = int(mb.chan[ch].FFT.FFT_SHIFT)
shift_factor = 2.0 ** bin(fft_shift).count('1')
model = mb.chan[ch].FFT.pfb_fft(frames, N=4)[:, :nbins] / shift_factor    # verify on hardware: absolute scale
model_pow = (np.abs(model) ** 2).mean(axis=0)

plt.figure(figsize=(12, 5))
plt.semilogy(freq[1:], fft_pow[model_input, 1:], lw=0.7, label='firmware FFT (fft_2x capture)')
plt.semilogy(freq[1:], model_pow[1:], lw=0.7, alpha=0.8, label='PFB/FFT model of the ADC capture')
plt.xlabel('frequency [MHz]'); plt.ylabel('|X|$^2$'); plt.title(f'{labels[model_input]}: ADC-derived vs firmware spectrum')
plt.grid(); plt.legend()

## 4. Digital gains

The scaler multiplies each FFT bin by a per-bin gain, then keeps the top 4+4 bits of the
48+48-bit product: a 4-bit value `q = round(gain * X / 2**44)`, clipped to ±7. For Gaussian
noise the empirical target rms of `q` is `1.5*sqrt(2)` (CHIME), so

    gain(bin) = target * 2**44 / mean|X(bin)|

`chFPGA.set_gains` takes the gains in natural bin order and, when no log gain is given,
programs them in floating-point mode (11-bit mantissa, 5-bit exponent). Reading the table
back gives the gains actually applied. A flat gain (as in `crs_corr64_tests`) is simply
`set_channelizers_async(gains=1 << 25)`.

In [ ]:
TARGET_RMS = 1.5 * np.sqrt(2)
fft_mag = np.abs(fft).mean(axis=1)                       # mean |X| per input and bin
# Limit the gain dynamic range to 60 dB per input: the float-gain format allows 31 octaves of
# per-bin exponent spread, and a dead or notched bin would otherwise ask for an absurd gain.
fft_mag = np.maximum(fft_mag, 1e-3 * fft_mag.max(axis=1, keepdims=True))
gain_req = TARGET_RMS * 2**44 / fft_mag                  # (n_inputs, nbins), natural order

gain_set = np.zeros_like(gain_req)
for b, mb in enumerate(ca.ib):
    mb.set_gains(gain={ch: gain_req[b * nchan + ch] for ch in range(nchan)}, bank=0)
    for ch in range(nchan):
        gain_set[b * nchan + ch] = mb.chan[ch].SCALER.get_gain_table(bank=0, use_cache=True)
await ca.sync_async()

# Fallback used by crs_corr64_tests: one flat gain for every bin and input (then
# gain_set[:] = 1 << 25 for the prediction in section 6).
# await ca.set_channelizers_async(gains=1 << 25, sync=False); gain_set[:] = 1 << 25

plt.figure(figsize=(12, 4))
for i in inputs_to_plot:
    plt.semilogy(freq[1:], gain_req[i, 1:], lw=0.7, label=f'{labels[i]} requested')
    plt.semilogy(freq[1:], gain_set[i, 1:], 'k--', lw=0.5, alpha=0.6)
plt.xlabel('frequency [MHz]'); plt.ylabel('scaler gain'); plt.title('digital gains (dashed: as programmed)')
plt.grid(); plt.legend(fontsize=8)

## 5. Correlator and visibilities

`integ_period` frames are accumulated in firmware; `soft_integ_period` of those results are
summed in software. Each board's correlator handles a subset of the bins and the bin set
rotates every firmware integration, so `soft_integ_period=8` collects every bin
(see `cr.get_good_bin_mask()` for the bins that carry data; a few are sacrificed for the
accumulator readout). `cr.get_ij_to_prod_map()[i, j]` is the product index of inputs i, j.

In [ ]:
integ_period = 32768                 # firmware frames per integration (8192..32768)
soft_integ_period = 8                # firmware integrations summed in software
await ca.start_correlators_async(integ_period)

cr = await ca.get_corr_receiver_async()
ijmap = cr.get_ij_to_prod_map()
good = cr.get_good_bin_mask()
n_inputs_corr = ijmap.shape[0]
print(f'correlator: {n_inputs_corr} inputs, {ijmap.max()+1} products, {good.sum()} of {good.size} bins carry data')

for b in ca.ib:                      # clear the sticky overrun flags before reading
    b.CORR.OVERRUN_RESET = 1; b.CORR.OVERRUN_RESET = 0
    b.UCAP.USER_RESET = 1;    b.UCAP.USER_RESET = 0

t_vis, vis, vis_cnt, vis_sat = cr.read_corr_frames(flush=True, soft_integ_period=soft_integ_period,
                                                   number_of_results=2, verbose=1)
print(f'visibilities {vis.shape} (results, bins, products), counts={vis_cnt}, saturations Re/Im={int(vis_sat.real.sum())}/{int(vis_sat.imag.sum())}')
print('overruns  UCAP:', list(ca.ib.UCAP.OVERRUN), ' CORR:', list(ca.ib.CORR.OVERRUN), '(all 0 expected)')

In [ ]:
# autocorrelations of a few inputs and one cross-correlation (amplitude and phase)
i0, j0 = inputs_to_plot[0], inputs_to_plot[1]
v = vis[0]
fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for i in inputs_to_plot:
    ax0.semilogy(freq[good], np.abs(v[good, ijmap[i, i]]), '.', ms=2, label=f'auto {labels[i]}')
ax0.semilogy(freq[good], np.abs(v[good, ijmap[i0, j0]]), '.', ms=2, label=f'cross ({i0},{j0})')
ax0.set_ylabel('|V|'); ax0.grid(); ax0.legend(fontsize=8); ax0.set_title('visibilities, first result')
ax1.plot(freq[good], np.degrees(np.angle(v[good, ijmap[i0, j0]])), '.', ms=2)
ax1.set(xlabel='frequency [MHz]', ylabel='phase [deg]', title=f'cross-correlation phase ({labels[i0]}) x ({labels[j0]})'); ax1.grid()

## 6. Overplot: ADC model, firmware FFT, quantized prediction, correlator

Top: pre-gain spectra (|X|²) from the ADC-derived model and the firmware FFT.
Bottom: what the correlator should see, predicted by applying the programmed gains to the
captured FFT and quantizing to 4+4 bits, against the measured autocorrelation divided by the
number of accumulated frames. `crs_corr64_tests` used an empirical factor 923/1031 instead
of the frame count here; if the two curves differ by a constant, that is the place to look.

*Verify on hardware:* the plot assumes correlator input `k` is board `ca.ib[k // 8]`,
channel `k % 8`. The reference notebook only ever compared input 0 with board 0 channel 0.
If input 0 agrees and inputs 1..3 do not, that mapping is the first suspect.

In [ ]:
n_frames_acc = integ_period * soft_integ_period                      # verify on hardware: normalization

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax0.semilogy(freq[1:], model_pow[1:], lw=0.6, alpha=0.7, label=f'ADC -> PFB/FFT model, {labels[model_input]}')
for i in inputs_to_plot:
    ax0.semilogy(freq[1:], fft_pow[i, 1:], lw=0.6, label=f'firmware FFT, {labels[i]}')
ax0.set_ylabel('|X|$^2$ (pre-gain)'); ax0.grid(); ax0.legend(fontsize=8)

for i in inputs_to_plot:
    q = np.round(fft[i] * gain_set[i] / 2**44)                        # 4+4-bit quantization of the captured FFT
    q = q.real.clip(-7, 7) + 1j * q.imag.clip(-7, 7)
    q_pow = (np.abs(q) ** 2).mean(axis=0)
    auto = np.abs(vis[0][:, ijmap[i, i]]) / n_frames_acc
    ax1.plot(freq[good], q_pow[good], '.', ms=2, label=f'quantized FFT prediction, {labels[i]}')
    ax1.plot(freq[good], auto[good], '+', ms=3, alpha=0.7, label=f'correlator autocorr / frames, {labels[i]}')
ax1.set(xlabel='frequency [MHz]', ylabel='4-bit power per frame'); ax1.grid(); ax1.legend(fontsize=8)

## Housekeeping

Corner-turn alignment monitors (corr32/corr64 only: the CT engines shuffle data between
boards; all zeros means no shuffling errors) and shutdown.

In [ ]:
if MODE != 'corr8':
    for b in ca.ib:
        for ct in (b.CT.CT2, b.CT.CT3):
            ct.ALIGN_MON_RESET = 1; ct.ALIGN_MON_RESET = 0; ct.ALIGN_MON_SOURCE = 0
    time.sleep(0.3)
    print('CT2 align errors:', [int(b.CT.CT2.ALIGN_MON_WORD) for b in ca.ib])
    print('CT3 align errors:', [int(b.CT.CT3.ALIGN_MON_WORD) for b in ca.ib])

In [ ]:
# stop the data flow (leave the boards programmed)
for b in ca.ib:
    b.stop_data_capture()
    b.CORR.stop_correlator()